In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
X_train  = train.drop(columns=['song_popularity', 'id'])
Y_train  = train['song_popularity']
X_test = test.drop(columns=['id'])

In [ ]:
X_train.head()

In [ ]:
categorical_features = {'key', 'audio_mode', 'time_signature'}
numerical_features = set(X_train.columns) - categorical_features
features = list(X_train.columns)

In [ ]:
# Handle missing values
X_train['song_duration_ms'].fillna(value=X_train['song_duration_ms'].median(), inplace=True)
X_train['acousticness'].fillna(value=X_train['acousticness'].median(), inplace=True)
X_train['danceability'].fillna(value=X_train['danceability'].median(), inplace=True)
X_train['energy'].fillna(value=X_train['energy'].median(), inplace=True)
X_train['instrumentalness'].fillna(value=X_train['instrumentalness'].median(), inplace=True)
X_train['key'].fillna(value=X_train['key'].median(), inplace=True)
X_train['liveness'].fillna(value=X_train['liveness'].median(), inplace=True)
X_train['loudness'].fillna(value=X_train['loudness'].median(), inplace=True)

X_test['song_duration_ms'].fillna(value=X_train['song_duration_ms'].median(), inplace=True)
X_test['acousticness'].fillna(value=X_train['acousticness'].median(), inplace=True)
X_test['danceability'].fillna(value=X_train['danceability'].median(), inplace=True)
X_test['energy'].fillna(value=X_train['energy'].median(), inplace=True)
X_test['instrumentalness'].fillna(value=X_train['instrumentalness'].median(), inplace=True)
X_test['key'].fillna(value=X_train['key'].median(), inplace=True)
X_test['liveness'].fillna(value=X_train['liveness'].median(), inplace=True)
X_test['loudness'].fillna(value=X_train['loudness'].median(), inplace=True)

In [ ]:
X_train.isnull().sum()

In [ ]:
from sklearn import preprocessing
scaler = preprocessing.MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_train.columns)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Plot combined histograms for each numerical feature colored by label
import matplotlib.pyplot as plt
import seaborn as sns

features_to_plot = list(numerical_features)
plt.figure(figsize=(18, 24))
for i, feature in enumerate(features_to_plot):
    plt.subplot(5, 2, i + 1)
    sns.histplot(data=X_train_scaled.assign(song_popularity=Y_train), x=feature, hue="song_popularity", bins=30, kde=True, element="step", stat="density")
    plt.title(f"{feature} by song_popularity")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Compute covariance matrix
fig = plt.figure(figsize = (15, 15))
df = X_train_scaled.copy()
df['song_popularity'] = Y_train
df = df[[col for col in df if df[col].nunique() > 1]]
corr_ = df.corr()
mask = np.zeros_like(corr_)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr_, mask = mask, annot=True)
plt.show()

In [ ]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import json

X, y = X_train_scaled, Y_train

# ======================
# LightGBM Optimization
# ======================
def objective_lgb(trial):
    params = {
        "device": "gpu",  # Use "cpu" if you don't have a GPU or LightGBM GPU version installed
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "n_jobs": 1,
        "seed": 42,
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.3, 0.8), # Equivalent to colsample_bytree
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 0.9), # Equivalent to subsample
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 10.0), # Equivalent to reg_alpha
        "lambda_l2": trial.suggest_float("lambda_l2", 1.0, 50.0, log=True), # Equivalent to reg_lambda
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100), # Equivalent to min_child_weight
        "scale_pos_weight": (np.sum(y == 0) / np.sum(y == 1)),
        # "gamma": trial.suggest_float("gamma", 0.0, 5.0), # Equivalent to min_split_gain
    }

    dtrain = lgb.Dataset(X, label=y)
    
    # Note: LightGBM's cv uses callbacks for early stopping
    cv_results = lgb.cv(
        params,
        dtrain,
        num_boost_round=500,
        nfold=5,
        stratified=True,
        seed=42,
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )
    
    # The number of rounds is the length of the results list
    best_num_boost_round = len(cv_results["valid auc-mean"])
    trial.set_user_attr("best_num_boost_round", best_num_boost_round)
    
    # The score to maximize is the max of the 'valid auc-mean' list
    return max(cv_results["valid auc-mean"])

study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=50)
best_lgb_params = study_lgb.best_params
best_lgb_rounds = study_lgb.best_trial.user_attrs["best_num_boost_round"]

print("Best LGBM params:", best_lgb_params)
print("Best LGBM rounds:", best_lgb_rounds)

# save the LGBM params and rounds to a file
with open('best_lgb_params.json', 'w') as f:
    json.dump(best_lgb_params, f)         
with open('best_lgb_rounds.json', 'w') as f:
    json.dump(best_lgb_rounds, f)

In [ ]:
import optuna
import xgboost as xgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X, y = X_train_scaled, Y_train

# ======================
# XGBoost Optimization
# ======================
def objective_xgb(trial):
    params = {
        "tree_method": "gpu_hist",
        "device": "cuda",
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "booster": "gbtree",
        "verbosity": 0,
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 50.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "scale_pos_weight": (y == 0).sum() / (y == 1).sum(),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }

    dtrain = xgb.DMatrix(X, label=y)
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=500,
        nfold=5,
        metrics="auc",
        early_stopping_rounds=20,
        seed=42,
        stratified=True,
    )
    trial.set_user_attr("best_num_boost_round", len(cv_results))
    return cv_results["test-auc-mean"].max()

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=50)
best_xgb_params = study_xgb.best_params
best_xgb_rounds = study_xgb.best_trial.user_attrs["best_num_boost_round"]

print("Best XGB params:", best_xgb_params)
print("Best XGB rounds:", best_xgb_rounds)

# save the XGB params and rounds to a file
import json
with open('best_xgb_params.json', 'w') as f:
    json.dump(best_xgb_params, f)
with open('best_xgb_rounds.json', 'w') as f:
    json.dump(best_xgb_rounds, f)

In [ ]:
best_lgb_params

In [ ]:
import optuna
import xgboost as xgb
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import json

# Assume X, y are pandas DataFrame/Series
X, y = X_train_scaled, Y_train

# ==========================================================
# Step 1: Generate Out-of-Fold (OOF) predictions once
# ==========================================================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize arrays to store OOF predictions
oof_preds_xgb = np.zeros(len(X))
oof_preds_lgb = np.zeros(len(X))

print("Generating out-of-fold predictions...")
for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y)):
    print(f"--- Fold {fold+1}/5 ---")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]

    # XGBoost
    dtrain_xgb = xgb.DMatrix(X_train, label=y_train)
    dvalid_xgb = xgb.DMatrix(X_valid, label=y_valid)
    xgb_fold = xgb.train(best_xgb_params, dtrain_xgb, num_boost_round=best_xgb_rounds)
    oof_preds_xgb[valid_idx] = xgb_fold.predict(dvalid_xgb)

    # LightGBM
    lgb_fold = lgb.LGBMClassifier(**best_lgb_params, n_estimators=best_lgb_rounds, n_jobs=-1, verbose=-1)
    lgb_fold.fit(X_train, y_train)
    oof_preds_lgb[valid_idx] = lgb_fold.predict_proba(X_valid)[:, 1]

print("OOF prediction generation complete.")
#print AUC scores for individual models
auc_xgb = roc_auc_score(y, oof_preds_xgb)
auc_lgb = roc_auc_score(y, oof_preds_lgb)
print("XGBoost OOF AUC:", auc_xgb)
print("LightGBM OOF AUC:", auc_lgb)

# ==========================================================
# Step 2: Optimize weights on the OOF predictions (very fast)
# ==========================================================
def objective_weights(trial):
    # Suggest two raw weights
    w_xgb_raw = trial.suggest_float("w_xgb_raw", 0.1, 1.0) # Avoid zero to ensure both models contribute as sometimes optuna assigns all of the weight to lgbm
    w_lgb_raw = trial.suggest_float("w_lgb_raw", 0.0, 1.0)

    # Normalize the weights to sum to 1
    total_weight = w_xgb_raw + w_lgb_raw
    w_xgb = w_xgb_raw / total_weight
    w_lgb = w_lgb_raw / total_weight
    
    # Calculate the ensembled predictions
    preds_ensemble = (w_xgb * oof_preds_xgb + 
                      w_lgb * oof_preds_lgb)
    
    # The objective is the AUC of the ensembled OOF predictions
    return roc_auc_score(y, preds_ensemble)

study_weights = optuna.create_study(direction="maximize")
study_weights.optimize(objective_weights, n_trials=100)  # Can run many trials since it's fast

# Calculate final normalized weights from the best trial
best_raw_weights = study_weights.best_params
total_best_weight = sum(best_raw_weights.values())
best_normalized_weights = {model.replace('_raw', ''): weight / total_best_weight 
                           for model, weight in best_raw_weights.items()}

print("\n--- Ensemble Results ---")
print("Best OOF AUC:", study_weights.best_value)
print("Best raw weights:", best_raw_weights)
print("Best normalized weights:", best_normalized_weights)


In [ ]:
best_weights = best_normalized_weights

In [ ]:
import xgboost as xgb

# Create DMatrix
dtrain_full = xgb.DMatrix(X_train_scaled, label=Y_train)

# Train final XGB model
final_xgb = xgb.train(
    best_xgb_params,
    dtrain_full,
    num_boost_round=best_xgb_rounds
)
dtest = xgb.DMatrix(X_test_scaled)
preds_xgb = final_xgb.predict(dtest)

In [ ]:
import lightgbm as lgb

final_lgb = lgb.LGBMClassifier(
    **best_lgb_params,
    n_estimators=best_lgb_rounds,
    n_jobs=-1,
    verbose=-1
)
final_lgb.fit(X_train_scaled, Y_train)
preds_lgb = final_lgb.predict_proba(X_test_scaled)[:, 1]

In [ ]:
w_xgb = best_weights['w_xgb']
w_lgb = best_weights['w_lgb']
preds_ensemble = w_xgb * preds_xgb + w_lgb * preds_lgb

In [ ]:
submission = pd.DataFrame({'id': test['id'], 'song_popularity': preds_ensemble})
submission.to_csv('submission.csv', index=False)